In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score
import joblib

# Load dataset
# Using on_bad_lines='skip' to bypass the error-causing rows in the CSV.
df = pd.read_csv("/content/emails.csv", on_bad_lines='skip')

# 'Prediction' is the label column, 1=spam, 0=not-spam
X = df.drop(columns=['Email No.', 'Prediction'])
y = df['Prediction']

# Clean and format data
X = X.apply(pd.to_numeric, errors='coerce').fillna(0)
y = y.astype(int)

# Split the data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# MLP classifier setup
# Hidden layers: (128, 64) nodes
# Max iterations: 200 (increased for better convergence)
clf = MLPClassifier(hidden_layer_sizes=(128,64), activation='relu', solver='adam', max_iter=200, random_state=42, verbose=False, early_stopping=True)

# Train the model
clf.fit(X_train, y_train)

# Evaluation
y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("--- Model Evaluation ---")
print("Accuracy:", accuracy)
print("F1:", f1)
print(classification_report(y_test, y_pred, target_names=['not-spam','spam']))

# Save model
joblib.dump(clf, "mlp_spam_bow.joblib")
print("Saved mlp_spam_bow.joblib")



--- Model Evaluation ---
Accuracy: 0.9797101449275363
F1: 0.9649415692821369
              precision    recall  f1-score   support

    not-spam       0.99      0.99      0.99       735
        spam       0.97      0.96      0.96       300

    accuracy                           0.98      1035
   macro avg       0.98      0.97      0.98      1035
weighted avg       0.98      0.98      0.98      1035

Saved mlp_spam_bow.joblib


In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertForSequenceClassification
# Corrected Imports for BERT Optimization
from torch.optim import AdamW
from transformers.optimization import get_linear_schedule_with_warmup
from sklearn.metrics import classification_report, accuracy_score, f1_score
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler

# --- Configuration ---
FILE_PATH = "emails (2).csv"
# ASSUMPTION based on file snippet: Column names are 'text' and 'spam'
TEXT_COLUMN = "text"
LABEL_COLUMN = "spam"
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5

# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- Load Data ---
try:
    df = pd.read_csv(FILE_PATH, encoding='latin-1', on_bad_lines='skip')

    # Filter columns to only keep the text and label columns and handle NaNs
    df = df[[TEXT_COLUMN, LABEL_COLUMN]].dropna().reset_index(drop=True)

    # Ensure labels are integers (0 or 1)
    df[LABEL_COLUMN] = df[LABEL_COLUMN].astype(int)
    print(f"Dataset loaded. Total samples: {len(df)}")

except KeyError as e:
    print(f"Error: Column {e} not found. Ensure your file has a '{TEXT_COLUMN}' and '{LABEL_COLUMN}' column.")
    # Print available columns to aid in manual correction
    print("Available columns in file:", pd.read_csv(FILE_PATH, encoding='latin-1', on_bad_lines='skip').columns.tolist())
    exit()
except Exception as e:
    print(f"An error occurred during data loading: {e}")
    exit()

# --- Split Data ---
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df[TEXT_COLUMN].tolist(), df[LABEL_COLUMN].tolist(),
    test_size=0.2, random_state=42, stratify=df[LABEL_COLUMN]
)
print(f"Train samples: {len(train_texts)}, Validation samples: {len(val_texts)}")

# --- Tokenization and BERT Input Formatting ---
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)

def encode_data(tokenizer, texts, max_len):
    return tokenizer.batch_encode_plus(
        texts,
        add_special_tokens=True,
        max_length=max_len,
        padding='max_length',
        return_attention_mask=True,
        return_tensors='pt',
        truncation=True
    )

# Encode data and create DataLoader instances... (The rest of this section remains the same)
train_encodings = encode_data(tokenizer, train_texts, MAX_LEN)
train_dataset = TensorDataset(train_encodings['input_ids'], train_encodings['attention_mask'], torch.tensor(train_labels))
train_dataloader = DataLoader(train_dataset, sampler=RandomSampler(train_dataset), batch_size=BATCH_SIZE)

val_encodings = encode_data(tokenizer, val_texts, MAX_LEN)
val_dataset = TensorDataset(val_encodings['input_ids'], val_encodings['attention_mask'], torch.tensor(val_labels))
validation_dataloader = DataLoader(val_dataset, sampler=SequentialSampler(val_dataset), batch_size=BATCH_SIZE)


# --- Model Setup ---
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels = 2,
    output_attentions = False,
    output_hidden_states = False,
)
model.to(device)

# Optimizer and Scheduler
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, eps=1e-8)
total_steps = len(train_dataloader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

# --- Training Loop ---
print("\nStarting BERT fine-tuning...")

for epoch_i in range(0, EPOCHS):
    # Training
    total_train_loss = 0
    model.train()
    # ... Training steps (identical to your original code) ...
    for step, batch in enumerate(train_dataloader):
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device)

        model.zero_grad()

        outputs = model(b_input_ids, token_type_ids=None, attention_mask=b_input_mask, labels=b_labels)

        loss = outputs.loss
        total_train_loss += loss.item()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

    avg_train_loss = total_train_loss / len(train_dataloader)
    print(f"  Epoch {epoch_i + 1}/{EPOCHS}: Average training loss: {avg_train_loss:.4f}")

# --- Evaluation ---
print("\nStarting BERT evaluation...")
model.eval()
predictions , true_labels = [], []

for batch in validation_dataloader:
    # ... Evaluation steps (identical to your original code) ...
    batch = tuple(t.to(device) for t in batch)
    b_input_ids, b_input_mask, b_labels = batch

    with torch.no_grad():
        outputs = model(b_input_ids, token_type_ids=None, attention_mask=b_input_mask)

    logits = outputs.logits
    logits = logits.detach().cpu().numpy()
    label_ids = b_labels.to('cpu').numpy()

    predictions.append(logits)
    true_labels.append(label_ids)

# Final Metrics
flat_predictions = np.argmax(np.concatenate(predictions, axis=0), axis=1).flatten()
flat_true_labels = np.concatenate(true_labels, axis=0)

accuracy = accuracy_score(flat_true_labels, flat_predictions)
f1 = f1_score(flat_true_labels, flat_predictions)

print("\n--- BERT Model Evaluation ---")
print("Accuracy:", accuracy)
print("F1:", f1)
print(classification_report(flat_true_labels, flat_predictions, target_names=['not-spam','spam']))

# --- Save Model ---
print("\nBERT fine-tuning and evaluation complete.")
# Remember to save the model and tokenizer state dictionary when running this code.



Using device: cuda
Dataset loaded. Total samples: 5728
Train samples: 4582, Validation samples: 1146


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Starting BERT fine-tuning...
  Epoch 1/3: Average training loss: 0.1102
  Epoch 2/3: Average training loss: 0.0143
  Epoch 3/3: Average training loss: 0.0017

Starting BERT evaluation...

--- BERT Model Evaluation ---
Accuracy: 0.9947643979057592
F1: 0.9890510948905109
              precision    recall  f1-score   support

    not-spam       1.00      1.00      1.00       872
        spam       0.99      0.99      0.99       274

    accuracy                           0.99      1146
   macro avg       0.99      0.99      0.99      1146
weighted avg       0.99      0.99      0.99      1146


BERT fine-tuning and evaluation complete.


#Updated

In [ ]:
import pandas as pd
from google.colab import files
import io

# Upload files
print("Upload your CSV files:")
uploaded = files.upload()

# Get file names
file_names = list(uploaded.keys())
print(f"Uploaded: {file_names}")

# Read all files
dataframes = []
for filename, content in uploaded.items():
    df = pd.read_csv(io.BytesIO(content))
    dataframes.append(df)
    print(f"📁 {filename}: {df.shape}")

# Combine horizontally
combined = pd.concat(dataframes, axis=1)
print(f"\n✅ Combined shape: {combined.shape}")

# Save and download
combined.to_csv('combined_files.csv', index=False)
files.download('combined_files.csv')

print("📥 combined_files.csv downloaded!")

Upload your CSV files:


Saving emails (2).csv to emails (2) (1).csv
Uploaded: ['emails (2) (1).csv']
📁 emails (2) (1).csv: (5728, 2)

✅ Combined shape: (5728, 2)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 combined_files.csv downloaded!


In [ ]:
# ============================================================
#  Spam Detection using Hybrid BERT + MLP Model
#  Inspired by "Spam Detection Using an Advanced Hybrid Model"
# ============================================================

import pandas as pd
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, random_split
from transformers import BertTokenizer, BertModel
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report

# ---------------- CONFIGURATION ----------------
FILE_PATH = "/content/combined_files.csv"  # <-- Change this to your dataset file
TEXT_COLUMN = "text"      # <-- Text column name
LABEL_COLUMN = "spam"     # <-- Label column name (0 = ham, 1 = spam)
MAX_LEN = 128
BATCH_SIZE = 16
EMBED_BATCH_SIZE = 32
EPOCHS = 5
LEARNING_RATE = 1e-4
DROPOUT_RATE = 0.3

# GPU/CPU Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on device: {device}")

# ---------------- LOAD DATA ----------------
df = pd.read_csv(FILE_PATH, on_bad_lines='skip', encoding='latin-1').dropna(subset=[TEXT_COLUMN, LABEL_COLUMN])
df[LABEL_COLUMN] = df[LABEL_COLUMN].astype(int)
print(f"Loaded {len(df)} samples.")

texts = df[TEXT_COLUMN].tolist()
labels = df[LABEL_COLUMN].tolist()

# ---------------- LOAD BERT TOKENIZER & MODEL ----------------
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased')
bert_model.to(device)
bert_model.eval()  # freeze BERT (used as feature extractor)

# ---------------- GENERATE BERT EMBEDDINGS ----------------
def get_bert_embeddings(texts):
    embeddings = []
    for i in range(0, len(texts), EMBED_BATCH_SIZE):
        batch_texts = texts[i:i+EMBED_BATCH_SIZE]
        encoded = tokenizer(batch_texts, padding=True, truncation=True, max_length=MAX_LEN, return_tensors='pt')
        input_ids = encoded['input_ids'].to(device)
        attention_mask = encoded['attention_mask'].to(device)

        with torch.no_grad():
            outputs = bert_model(input_ids, attention_mask=attention_mask)
        cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()  # CLS token
        embeddings.extend(cls_embeddings)
    return np.array(embeddings)

print("Extracting BERT embeddings...")
X = get_bert_embeddings(texts)
y = np.array(labels)
print("Embeddings shape:", X.shape)

# ---------------- SPLIT DATA ----------------
split_ratio = 0.8
train_size = int(split_ratio * len(X))
val_size = len(X) - train_size

X_train, X_val = torch.tensor(X[:train_size]).float(), torch.tensor(X[train_size:]).float()
y_train, y_val = torch.tensor(y[:train_size]).float(), torch.tensor(y[train_size:]).float()

train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

# ---------------- DEFINE MLP MODEL ----------------
class MLPClassifier(nn.Module):
    def __init__(self, input_dim=768, hidden1=256, hidden2=128, dropout_rate=0.3):
        super(MLPClassifier, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden2, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

mlp_model = MLPClassifier(dropout_rate=DROPOUT_RATE).to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(mlp_model.parameters(), lr=LEARNING_RATE)

# ---------------- TRAINING LOOP ----------------
print("\nStarting MLP training...")
for epoch in range(EPOCHS):
    mlp_model.train()
    total_loss = 0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device).unsqueeze(1)
        optimizer.zero_grad()
        outputs = mlp_model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Training Loss: {avg_loss:.4f}")

# ---------------- EVALUATION ----------------
mlp_model.eval()
predictions, true_labels = [], []

with torch.no_grad():
    for batch_x, batch_y in val_loader:
        batch_x = batch_x.to(device)
        outputs = mlp_model(batch_x)
        preds = (outputs.cpu().numpy() > 0.5).astype(int).flatten()
        predictions.extend(preds)
        true_labels.extend(batch_y.numpy())

accuracy = accuracy_score(true_labels, predictions)
f1 = f1_score(true_labels, predictions)
roc = roc_auc_score(true_labels, predictions)
print("\n--- Hybrid BERT + MLP Evaluation ---")
print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"ROC-AUC: {roc:.4f}")
print(classification_report(true_labels, predictions, target_names=["ham","spam"]))

# ---------------- SAVE MODELS ----------------
torch.save(mlp_model.state_dict(), "hybrid_mlp_model.pt")
bert_model.save_pretrained("./bert_feature_extractor")
tokenizer.save_pretrained("./bert_feature_extractor")

print("\n✅ Hybrid BERT + MLP model training complete and saved successfully!")

Running on device: cuda
Loaded 5728 samples.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Extracting BERT embeddings...
Embeddings shape: (5728, 768)

Starting MLP training...
Epoch [1/5] - Training Loss: 0.3857
Epoch [2/5] - Training Loss: 0.1331
Epoch [3/5] - Training Loss: 0.0865
Epoch [4/5] - Training Loss: 0.0719
Epoch [5/5] - Training Loss: 0.0596

--- Hybrid BERT + MLP Evaluation ---
Accuracy: 0.9852
F1 Score: 0.0000
ROC-AUC: nan
              precision    recall  f1-score   support

         ham       1.00      0.99      0.99      1146
        spam       0.00      0.00      0.00         0

    accuracy                           0.99      1146
   macro avg       0.50      0.49      0.50      1146
weighted avg       1.00      0.99      0.99      1146



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zer


✅ Hybrid BERT + MLP model training complete and saved successfully!


In [ ]:
# ============================================================
#  Hybrid Spam Detector (BERT + Numeric MLP)
#  Aligned to your emails.csv + emails (2).csv
# ============================================================

import pandas as pd
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import BertTokenizer, BertModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report

# ---------------- CONFIG ----------------
FILE_TEXT = "/content/emails (2).csv"   # Text + spam labels
FILE_NUMERIC = "/content/emails.csv"    # Numeric word frequencies
TEXT_COLUMN = "text"
LABEL_COLUMN = "spam"
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 5
LEARNING_RATE = 1e-4
DROPOUT_RATE = 0.3
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on device: {device}")

# ---------------- LOAD DATA ----------------
df_text = pd.read_csv(FILE_TEXT, on_bad_lines='skip', encoding='latin-1')
df_num = pd.read_csv(FILE_NUMERIC, on_bad_lines='skip', encoding='latin-1')

# Clean and align
min_len = min(len(df_text), len(df_num))
df_text = df_text.head(min_len).dropna(subset=[TEXT_COLUMN, LABEL_COLUMN]).reset_index(drop=True)
df_num = df_num.head(min_len).reset_index(drop=True)

# Labels
y = df_text[LABEL_COLUMN].astype(int).values

# ---------------- BERT EMBEDDINGS ----------------
print("Loading BERT...")
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased')
bert_model.to(device)
bert_model.eval()

def get_bert_embeddings(texts):
    embeddings = []
    for i in range(0, len(texts), 32):
        batch_texts = texts[i:i+32]
        encoded = tokenizer(batch_texts, padding=True, truncation=True, max_length=MAX_LEN, return_tensors='pt')
        input_ids = encoded['input_ids'].to(device)
        attention_mask = encoded['attention_mask'].to(device)
        with torch.no_grad():
            outputs = bert_model(input_ids, attention_mask=attention_mask)
        cls_emb = outputs.last_hidden_state[:, 0, :].cpu().numpy()  # CLS token
        embeddings.extend(cls_emb)
    return np.array(embeddings)

print("Generating BERT embeddings from email subjects...")
X_text = get_bert_embeddings(df_text[TEXT_COLUMN].astype(str).tolist())
print("BERT embedding shape:", X_text.shape)

# ---------------- NUMERIC FEATURES ----------------
df_num_clean = df_num.drop(columns=['Email No.', 'Prediction'], errors='ignore')
X_num = df_num_clean.apply(pd.to_numeric, errors='coerce').fillna(0).values
print("Numeric features shape:", X_num.shape)

# ---------------- COMBINE ----------------
X_combined = np.concatenate([X_text, X_num], axis=1)
print("Combined feature shape:", X_combined.shape)

# ---------------- TRAIN / TEST SPLIT ----------------
X_train, X_val, y_train, y_val = train_test_split(
    X_combined, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val = torch.tensor(X_train).float(), torch.tensor(X_val).float()
y_train, y_val = torch.tensor(y_train).float(), torch.tensor(y_val).float()

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=BATCH_SIZE)

# ---------------- MLP CLASSIFIER ----------------
class HybridMLP(nn.Module):
    def __init__(self, input_dim, hidden1=512, hidden2=128, dropout=0.3):
        super(HybridMLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden2, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

mlp_model = HybridMLP(X_combined.shape[1], dropout=DROPOUT_RATE).to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(mlp_model.parameters(), lr=LEARNING_RATE)

# ---------------- TRAINING ----------------
print("\nTraining hybrid MLP...")
for epoch in range(EPOCHS):
    mlp_model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device).unsqueeze(1)
        optimizer.zero_grad()
        preds = mlp_model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS} - Avg Loss: {total_loss/len(train_loader):.4f}")

# ---------------- EVALUATION ----------------
mlp_model.eval()
preds, labels = [], []
with torch.no_grad():
    for xb, yb in val_loader:
        xb = xb.to(device)
        outputs = mlp_model(xb)
        preds.extend((outputs.cpu().numpy() > 0.5).astype(int).flatten())
        labels.extend(yb.numpy())

acc = accuracy_score(labels, preds)
f1 = f1_score(labels, preds)
roc = roc_auc_score(labels, preds)
print("\n--- Hybrid Model Evaluation ---")
print(f"Accuracy: {acc:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"ROC-AUC: {roc:.4f}")
print(classification_report(labels, preds, target_names=["ham","spam"]))

# ---------------- SAVE MODELS ----------------
torch.save(mlp_model.state_dict(), "hybrid_mlp_full.pt")
bert_model.save_pretrained("./bert_feature_extractor")
tokenizer.save_pretrained("./bert_feature_extractor")
print("\n✅ Model training complete. Models saved successfully!")

Running on device: cuda
Loading BERT...
Generating BERT embeddings from email subjects...
BERT embedding shape: (5172, 768)
Numeric features shape: (5172, 3000)
Combined feature shape: (5172, 3768)

Training hybrid MLP...
Epoch 1/5 - Avg Loss: 0.4712
Epoch 2/5 - Avg Loss: 0.2306
Epoch 3/5 - Avg Loss: 0.1266
Epoch 4/5 - Avg Loss: 0.1043
Epoch 5/5 - Avg Loss: 0.0804

--- Hybrid Model Evaluation ---
Accuracy: 0.9652
F1 Score: 0.9348
ROC-AUC: 0.9577
              precision    recall  f1-score   support

         ham       0.98      0.97      0.98       761
        spam       0.93      0.94      0.93       274

    accuracy                           0.97      1035
   macro avg       0.95      0.96      0.96      1035
weighted avg       0.97      0.97      0.97      1035


✅ Model training complete. Models saved successfully!


In [ ]:
# After training:
torch.save(mlp_model.state_dict(), "hybrid_mlp.pt")
bert_model.save_pretrained("bert_feature_extractor")
tokenizer.save_pretrained("bert_feature_extractor")

# Later, for prediction:
from transformers import BertTokenizer, BertModel
import torch.nn.functional as F

tokenizer = BertTokenizer.from_pretrained("bert_feature_extractor")
bert = BertModel.from_pretrained("bert_feature_extractor")
mlp.load_state_dict(torch.load("hybrid_mlp.pt"))

def predict_spam(text):
    encoded = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=128)
    with torch.no_grad():
        emb = bert(**encoded).last_hidden_state[:, 0, :]
        output = mlp(emb)
        return "SPAM" if output.item() > 0.5 else "HAM"

NameError: name 'mlp' is not defined

In [ ]:
import torch
import torch.nn as nn
from transformers import BertTokenizer, BertModel

# --- Reload BERT + Tokenizer ---
tokenizer = BertTokenizer.from_pretrained("bert_feature_extractor")
bert = BertModel.from_pretrained("bert_feature_extractor")
bert.eval()

# --- Redefine the same MLP architecture used during training ---
class HybridMLP(nn.Module):
    def __init__(self, input_dim=3768, hidden1=512, hidden2=128, dropout=0.3):
        super(HybridMLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden2, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

# --- Initialize model and load saved weights ---
mlp = HybridMLP()
mlp.load_state_dict(torch.load("hybrid_mlp_full.pt", map_location=torch.device('cpu')))
mlp.eval()

# --- Numeric feature helper (optional) ---
# If you want to test on a full email that includes numeric counts:
# you can pass zeros or average values here
import numpy as np
dummy_numeric = np.zeros((1, 3000))  # adjust dimension = number of numeric features you used

# --- Prediction Function ---
def predict_spam(text):
    # Step 1: Get BERT embedding
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=128)
    with torch.no_grad():
        outputs = bert(**inputs)
        bert_emb = outputs.last_hidden_state[:, 0, :].numpy()

    # Step 2: Combine BERT + numeric features
    combined_features = np.concatenate([bert_emb, dummy_numeric], axis=1)
    combined_tensor = torch.tensor(combined_features).float()

    # Step 3: Predict
    with torch.no_grad():
        pred = mlp(combined_tensor).item()
    return "SPAM" if pred > 0.5 else "HAM (Not Spam)", float(pred)

# --- Example Predictions ---
msg1 = "Congratulations! You have won a free iPhone. Click here to claim."
msg2 = "Hey, let's meet for lunch tomorrow at 1 PM."

print(f"1️⃣: {predict_spam(msg1)}")
print(f"2️⃣: {predict_spam(msg2)}")

1️⃣: ('SPAM', 0.9966369867324829)
2️⃣: ('HAM (Not Spam)', 0.045299142599105835)


In [ ]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 31.3 MB/s eta 0:00:00


In [ ]:
!pip install streamlit pyngrok

# Create a file for your app
%%writefile app.py
import streamlit as st
st.title("✅ Streamlit Test")
st.write("If you can see this, Streamlit works inside Colab!")

# Now run the Streamlit app with ngrok
from pyngrok import ngrok
!streamlit run app.py &>/dev/null&
public_url = ngrok.connect(8501)
print("Public URL:", public_url)

UsageError: Line magic function `%%writefile` not found.


In [ ]:
# ============================================================
# SpamGuardian Mail - Interactive Gmail-like Demo
# ============================================================
import streamlit as st
import torch
import torch.nn as nn
import numpy as np
from transformers import BertTokenizer, BertModel

# --- Load tokenizer, model ---
tokenizer = BertTokenizer.from_pretrained("bert_feature_extractor")
bert = BertModel.from_pretrained("bert_feature_extractor")

# --- Define your MLP model ---
class HybridMLP(nn.Module):
    def __init__(self, input_dim=3768, hidden1=512, hidden2=128, dropout=0.3):
        super(HybridMLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden2, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

mlp = HybridMLP()
mlp.load_state_dict(torch.load("hybrid_mlp_full.pt", map_location=torch.device('cpu')))
mlp.eval()

# --- Streamlit UI Layout ---
st.set_page_config(page_title="SpamGuardian Mail", layout="wide")
st.title("📧 SpamGuardian Mail")

col1, col2 = st.columns(2)

with col1:
    st.subheader("✉️ Compose Email")
    sender = st.text_input("From:", "user@example.com")
    receiver = st.text_input("To:", "friend@example.com")
    subject = st.text_input("Subject:", "")
    body = st.text_area("Message:", "", height=200)
    send_button = st.button("Send Mail")

with col2:
    st.subheader("📥 Receiver Inbox")
    if send_button:
        full_message = f"Subject: {subject}\n\n{body}"
        # Generate BERT embedding
        encoded = tokenizer(full_message, return_tensors='pt', truncation=True, padding=True, max_length=128)
        with torch.no_grad():
            emb = bert(**encoded).last_hidden_state[:, 0, :].numpy()

        # Dummy numeric vector (zeros)
        numeric_dummy = np.zeros((1, 3000))
        combined = np.concatenate([emb, numeric_dummy], axis=1)
        combined_tensor = torch.tensor(combined).float()

        with torch.no_grad():
            score = mlp(combined_tensor).item()

        if score > 0.5:
            st.error("🚫 SPAM detected! This email was blocked by AI filter.")
            st.markdown(f"**Spam Probability:** {score:.2f}")
        else:
            st.success("✅ Safe Mail delivered successfully!")
            st.markdown(f"**Spam Probability:** {score:.2f}")
            st.write("---")
            st.write(f"**From:** {sender}")
            st.write(f"**To:** {receiver}")
            st.write(f"**Subject:** {subject}")
            st.write(body)

2025-11-13 06:48:19.141 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-13 06:48:19.142 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-13 06:48:19.143 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-13 06:48:19.145 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-13 06:48:19.147 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-13 06:48:19.148 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-13 06:48:19.149 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-13 06:48:19.150 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [ ]:
%%writefile app.py
import streamlit as st

st.set_page_config(page_title="SpamGuardian Demo", layout="centered")

st.title("📧 SpamGuardian – AI Spam Detection Demo")

st.markdown("### Compose an Email")
sender = st.text_input("From:")
subject = st.text_input("Subject:")
body = st.text_area("Email Body:", height=150)

if st.button("Send Email"):
    # Dummy example – replace this with your trained model later
    spam_keywords = ["win", "lottery", "free", "click", "urgent", "offer"]
    result = "spam" if any(word in body.lower() for word in spam_keywords) else "not spam"

    st.markdown("---")
    st.subheader("📨 Receiver Inbox View")
    if result == "spam":
        st.warning("⚠️ This email was filtered into the *Spam Folder*.")
    else:
        st.success("✅ This email arrived safely in the *Inbox*.")

Writing app.py


In [ ]:
from pyngrok import ngrok
ngrok.set_auth_token("35PfbeuNtKtl44sd0tjhnZOJFgU_3H5SbGmDuERhXcjv33qXL")
!streamlit run app.py &>/dev/null&
public_url = ngrok.connect(8501)
print("🌐 Public URL:", public_url)

🌐 Public URL: NgrokTunnel: "https://injurable-glossarial-kaycee.ngrok-free.dev" -> "http://localhost:8501"
